In [78]:
import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix


print("PyTorch version:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available())

PyTorch version: 2.14.1
MPS available: True


In [79]:
import pandas as pd

train_df= pd.read_csv("../final_data/train/dico_nli_train_track1_participant_labeled.csv")

dev_df=pd.read_csv(  "../final_data/dev/dico_nli_dev_track1_participant_labeled.csv")

print(train_df.shape)
print(dev_df.shape)

(3042, 7)
(660, 7)


In [80]:
label2id={
    "BACKWARD_ENTAILMENT":0,
    "EQUIVALENCE":1,
    "FORWARD_ENTAILMENT":2,
    "NEGATIVE_OTHER":3
}

id2label= {
    0: "BACKWARD_ENTAILMENT",
    1: "EQUIVALENCE",
    2: "FORWARD_ENTAILMENT",
    3: "NEGATIVE_OTHER"
}

In [81]:
train_df["label_id"]=train_df["label"].map(label2id)

dev_df["label_id"]=dev_df["label"].map(label2id)
# control: shouldnt be nan
print("NaN:", train_df["label_id"].isna().sum(), dev_df["label_id"].isna().sum())
print(train_df["label"].value_counts())
print(dev_df["label"].value_counts())

#train_df[["label","label_id"]].head()

NaN: 0 0
label
FORWARD_ENTAILMENT     881
BACKWARD_ENTAILMENT    881
EQUIVALENCE            802
NEGATIVE_OTHER         478
Name: count, dtype: int64
label
FORWARD_ENTAILMENT     190
BACKWARD_ENTAILMENT    190
EQUIVALENCE            174
NEGATIVE_OTHER         106
Name: count, dtype: int64


In [82]:
from transformers import AutoTokenizer

# model_name = "microsoft/deberta-v3-small"   # gives NaN loss on Mac (MPS) during training
model_name = "distilroberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded")

Tokenizer loaded


In [83]:
example=tokenizer("A brown and white dog","A dog")
print(example)

{'input_ids': [0, 250, 6219, 8, 1104, 2335, 2, 2, 250, 2335, 2], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


tokens=tokenizer.convert_ids_to_tokens(example["input_ids"])
print(tokens)

In [84]:
from datasets import Dataset

train_dataset= Dataset.from_pandas(train_df[["text1", "text2", "label_id"]])

dev_dataset= Dataset.from_pandas(dev_df[["text1", "text2", "label_id"]])

print(train_dataset)
print(dev_dataset)


Dataset({
    features: ['text1', 'text2', 'label_id'],
    num_rows: 3042
})
Dataset({
    features: ['text1', 'text2', 'label_id'],
    num_rows: 660
})


In [85]:
def tokenize_batch(batch):
    return tokenizer(
        batch["text1"],
        batch["text2"],
        truncation=True,
        max_length=128,     
    )



In [86]:


tokenized_train=train_dataset.map(
    tokenize_batch,
    batched=True
)
tokenized_dev=dev_dataset.map(
    tokenize_batch,
    batched=True
)

print(tokenized_train)


Map:   0%|          | 0/3042 [00:00<?, ? examples/s]

Map:   0%|          | 0/660 [00:00<?, ? examples/s]

Dataset({
    features: ['text1', 'text2', 'label_id', 'input_ids', 'attention_mask'],
    num_rows: 3042
})


In [87]:
tokenized_train = tokenized_train.rename_column("label_id", "labels")
tokenized_dev = tokenized_dev.rename_column("label_id", "labels")

print(tokenized_train)


Dataset({
    features: ['text1', 'text2', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 3042
})


In [88]:
from transformers import DataCollatorWithPadding

data_collator= DataCollatorWithPadding(tokenizer=tokenizer)
print(data_collator)

DataCollatorWithPadding(tokenizer=RobertaTokenizer(name_or_path='distilroberta-base', vocab_size=50265, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}, added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	50264: AddedToken("<mask>", rstrip=False, lstrip=True, single_word=False, normalized=True, special=True),
}), padding=True, max_length=None, pad_to_multiple_of=None, return_tensors='pt')


In [89]:
from transformers import AutoModelForSequenceClassification

model=AutoModelForSequenceClassification.from_pretrained( 
    model_name,
    num_labels=4, #bc we have 4 classes backward, forward, negative, equivalent
    id2label=id2label,
    label2id=label2id
)

print("model loaded")

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model loaded


In [90]:
device=torch.device("mps" if torch.backends.mps.is_available() else "cpu") 
#if macs gpu can be used with mps(metal perdormance shaders) (it is the name ofapples structure for using gpu in pytorch i guess ) use it, else go back to cpu

model.to(device)
print("using devive", device)

using devive mps


In [91]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(eval_pred):
    logits, labels= eval_pred
    preds=np.argmax(logits, axis=-1)

    return{
        "accuracy": accuracy_score(labels, preds),
       
        "weighted_f1":f1_score(labels, preds, average="weighted")
    }

#we are creating 4 scores to 4 classes for every example, and np.argmaax chooses the highest scores index as final prediction
#then we compare our predictions with real labels and calculate accuarcy and weightedf1

In [92]:
from transformers import TrainingArguments, Trainer

# training_args = TrainingArguments(
#     output_dir="../results/deberta_v3_small",
#     eval_strategy="epoch",
#     save_strategy="epoch",

#     learning_rate=2e-5,

#     per_device_train_batch_size=8,
#     per_device_eval_batch_size=8,

#     num_train_epochs=3,

#     weight_decay=0.01,

#     load_best_model_at_end=True,
#     metric_for_best_model="weighted_f1",
#     greater_is_better=True,

#     logging_steps=20
#     
# )


training_args = TrainingArguments(
    output_dir="../results/distilroberta",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,

    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    num_train_epochs=5,

    warmup_steps=95,
    weight_decay=0.01,
    max_grad_norm=1.0,

    load_best_model_at_end=True,
    metric_for_best_model="weighted_f1",
    greater_is_better=True,

    logging_steps=20,
    fp16=False,
    bf16=False,
    report_to="none",
    logging_nan_inf_filter=False,
)


# epoch: when model sees the train data 1 time from the beggining to the end
# our batch size is 8, so our model process train data (3042 examples) as 380 batches
# we have 3 epoches each of them seeing the train examples and calculates set scores
# it is not true that "more epoch better scores", bc model may start to memorize train set. Its called overfitting

In [93]:
from transformers import Trainer 
trainer= Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_dev,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("trainer ready")
print("device:", trainer.args.device)

trainer ready
device: mps


In [94]:
trainer.train()

/Users/sudenurgokce/Desktop/SemEval-2027-Task-2-DiCo-NLI/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Weighted F1
1,1.210557,1.255082,0.487879,0.462879
2,0.810071,1.067841,0.612121,0.593624
3,0.541749,1.021993,0.634848,0.629826
4,0.460606,1.141943,0.640909,0.631272
5,0.304225,1.172896,0.646970,0.641702


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/sudenurgokce/Desktop/SemEval-2027-Task-2-DiCo-NLI/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/Users/sudenurgokce/Desktop/SemEval-2027-Task-2-DiCo-NLI/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/sudenurgokce/Desktop/SemEval-2027-Task-2-DiCo-NLI/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=955, training_loss=0.7190906130206522, metrics={'train_runtime': 290.0, 'train_samples_per_second': 52.448, 'train_steps_per_second': 3.293, 'total_flos': 69393141480432.0, 'train_loss': 0.7190906130206522, 'epoch': 5.0})

In [95]:
print(tokenized_train.features)
print(tokenized_train[0]["labels"])
print(tokenized_train[0]["text1"])
print(tokenized_train[0]["text2"])

{'text1': Value('large_string'), 'text2': Value('large_string'), 'labels': Value('int64'), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}
2
Conservative opposition
Conservatives


In [96]:
from sklearn.metrics import classification_report, confusion_matrix

out = trainer.predict(tokenized_dev)
preds = np.argmax(out.predictions, axis=-1)
labels = out.label_ids

dev_df["pred"] = [id2label[i] for i in preds]   # keep predictions for error analysis later

print("predicted distribution:", np.bincount(preds, minlength=4))
print("true distribution:     ", np.bincount(labels, minlength=4))
print(classification_report(labels, preds, target_names=[id2label[i] for i in range(4)], digits=3))
print(confusion_matrix(labels, preds))

/Users/sudenurgokce/Desktop/SemEval-2027-Task-2-DiCo-NLI/.venv/lib/python3.14/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


predicted distribution: [223 141 195 101]
true distribution:      [190 174 190 106]
                     precision    recall  f1-score   support

BACKWARD_ENTAILMENT      0.668     0.784     0.722       190
        EQUIVALENCE      0.631     0.511     0.565       174
 FORWARD_ENTAILMENT      0.754     0.774     0.764       190
     NEGATIVE_OTHER      0.416     0.396     0.406       106

           accuracy                          0.647       660
          macro avg      0.617     0.616     0.614       660
       weighted avg      0.643     0.647     0.642       660

[[149  17   9  15]
 [ 31  89  21  33]
 [ 11  21 147  11]
 [ 32  14  18  42]]


In [97]:
transformer_submission = dev_df[["instance_id"]].copy()
transformer_submission["label"] = dev_df["pred"]

transformer_submission.to_csv("../submissions/transformer_distilroberta.csv", index=False)
transformer_submission.head()

,instance_id,label
0,dico_phrasis_0000002__en-en__flipped,FORWARD_ENTAILMENT
1,dico_phrasis_0000002__en-en__original,BACKWARD_ENTAILMENT
2,dico_phrasis_0000004__en-en__flipped,FORWARD_ENTAILMENT
3,dico_phrasis_0000004__en-en__original,BACKWARD_ENTAILMENT
4,dico_phrasis_0000008__en-en__flipped,FORWARD_ENTAILMENT
